# 06 — Systematic Riparian Hotspot Scan

**Goal:** notebook 05 found real riverside encroachment at Mathare and Kibera, but only because
those were checked by name — chosen from documented literature, not discovered by the pipeline.
This notebook removes that dependency: scan the whole city on a grid, flag cells where the
riverside built-up fraction is well above the immediate surrounding area, and see what surfaces
without prior knowledge of where to look.

**Method:** a 500m grid (`Geometry.coveringGrid`, UTM 37S) over Nairobi. For each cell, compare
built-up fraction in riverside pixels (within 30m of a river, per notebook 05's buffer) against
built-up fraction in the rest of that same cell. Cells with too few pixels on either side (< 20,
~2000 m²) are dropped — with a thin 30m buffer sliced through a 500m cell, small cells produce
noisy, meaningless 0%/100% swings that aren't a real signal.

**Why 500m, and what that choice costs:** small enough to point at specific stretches of river,
large enough to have a meaningful non-riverside sample inside the same cell. The trade-off (found
by cross-checking against notebook 05's known hotspots, below) is that this is closer to an *edge
detector* than a *density detector* — it flags places where the river marks a sharp local
boundary between built and unbuilt, and can miss settlements that are uniformly dense on both
sides of that 30m line, even if the settlement as a whole is a real riverside encroachment case
at a wider scale.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
from acquisition import get_nairobi_boundary, get_sentinel2_composite
from classification import (
    build_feature_image, get_worldcover_builtup, sample_training_points,
    train_random_forest, classify_builtup,
)

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
proj = ee.Projection('EPSG:32737')  # UTM 37S — correct hemisphere for Nairobi
grid = nairobi.coveringGrid(proj, 500)
grid = grid.map(lambda f: f.set('centroid', f.geometry().centroid(1).coordinates()))
print('Grid cells:', grid.size().getInfo())

rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers').filterBounds(nairobi)
dist_to_river = rivers.distance(searchRadius=200, maxError=10).clip(nairobi)
riverside_mask = dist_to_river.lte(30)

Grid cells: 3016


## Classify 2024 built-up (same procedure as notebooks 03/05)

In [2]:
composite, scene_count = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
features = build_feature_image(composite)
worldcover_builtup = get_worldcover_builtup(nairobi)
train_samples, test_samples = sample_training_points(features, worldcover_builtup, nairobi)
classifier = train_random_forest(train_samples)
builtup = classify_builtup(features, classifier)

test_accuracy = test_samples.classify(classifier).errorMatrix(
    'builtup', 'classification'
).accuracy().getInfo()
print(f'Scenes: {scene_count}, held-out accuracy (sanity check): {test_accuracy * 100:.1f}%')

Scenes: 11, held-out accuracy (sanity check): 85.9%


## Per-cell riverside vs. rest-of-cell built-up fraction

Two `reduceRegions` passes over the grid — one on riverside pixels only, one on everything
else — rather than a single grouped reducer, which turned out simpler to get reliable per-cell
output from than fighting `Reducer.group`'s field-indexing inside `reduceRegions`.

In [3]:
builtup_riverside = builtup.updateMask(riverside_mask).rename('b')
builtup_rest = builtup.updateMask(riverside_mask.Not()).rename('b')

reducer = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

stats_riverside = builtup_riverside.reduceRegions(collection=grid, reducer=reducer, scale=10).getInfo()['features']
stats_rest = builtup_rest.reduceRegions(collection=grid, reducer=reducer, scale=10).getInfo()['features']

MIN_PIXELS = 20
rows = []
for r, s in zip(stats_riverside, stats_rest):
    pr, ps = r['properties'], s['properties']
    if pr['count'] is None or pr['count'] < MIN_PIXELS:
        continue
    if ps['count'] is None or ps['count'] < MIN_PIXELS:
        continue
    lon, lat = pr['centroid']
    rows.append({
        'lon': lon, 'lat': lat,
        'riverside_pct': pr['mean'] * 100, 'riverside_n': pr['count'],
        'rest_pct': ps['mean'] * 100, 'rest_n': ps['count'],
        'diff_pp': (pr['mean'] - ps['mean']) * 100,
    })

rows.sort(key=lambda x: -x['diff_pp'])
print(f'{len(rows)} of {len(stats_riverside)} cells have enough pixels on both sides to compare.')

578 of 3016 cells have enough pixels on both sides to compare.


## Top hotspots — riverside built-up % well above the immediate surrounding area

In [4]:
print(f'{"lon":>9} {"lat":>9} {"riverside %":>12} {"rest %":>8} {"diff":>8}')
for row in rows[:15]:
    print(f"{row['lon']:>9.4f} {row['lat']:>9.4f} {row['riverside_pct']:>11.1f}% {row['rest_pct']:>7.1f}% {row['diff_pp']:>+7.1f}pp")

      lon       lat  riverside %   rest %     diff
  36.7960   -1.2815        86.7%    30.2%   +56.4pp
  36.9173   -1.2906        78.0%    23.2%   +54.8pp
  36.7960   -1.2499        96.1%    43.8%   +52.3pp
  36.8005   -1.2137        76.5%    26.2%   +50.3pp
  36.9308   -1.1686        82.1%    40.5%   +41.5pp
  36.8140   -1.2770        93.3%    53.9%   +39.4pp
  37.0610   -1.2772        79.5%    40.7%   +38.8pp
  37.0745   -1.2908        53.0%    14.9%   +38.1pp
  36.8904   -1.2002        78.6%    42.1%   +36.5pp
  36.8140   -1.2725       100.0%    65.4%   +34.6pp
  36.8904   -1.2364        57.7%    23.4%   +34.3pp
  36.9757   -1.2590        74.4%    40.4%   +34.0pp
  36.9308   -1.2409        99.7%    69.0%   +30.7pp
  36.9757   -1.2229        78.9%    49.4%   +29.5pp
  36.9308   -1.2183       100.0%    70.6%   +29.4pp


## Cross-check against notebook 05's known hotspots

Notebook 05 checked Mathare, Kibera, and Mukuru using a 1km-radius "surrounding" region. Here,
the "surrounding" region is only the rest of the same 500m cell — a much tighter neighborhood.
Re-running the same three locations through this grid's cell-level numbers tests whether the two
methods agree, and if not, why.

In [5]:
hotspots = {
    'Mathare': (36.857, -1.259),
    'Kibera': (36.789, -1.313),
    'Mukuru': (36.870, -1.310),
}

def nearest_cell(lon, lat):
    return min(rows_all, key=lambda r: (r['lon']-lon)**2 + (r['lat']-lat)**2)

rows_all = []
for r, s in zip(stats_riverside, stats_rest):
    pr, ps = r['properties'], s['properties']
    lon, lat = pr['centroid']
    rows_all.append({
        'lon': lon, 'lat': lat,
        'riverside_pct': (pr['mean'] or 0) * 100, 'riverside_n': pr['count'] or 0,
        'rest_pct': (ps['mean'] or 0) * 100, 'rest_n': ps['count'] or 0,
    })

print(f'{"Location":>10} {"Riverside %":>12} {"Rest-of-cell %":>15} {"Diff":>8}   (notebook 05: 1km-radius diff)')
notebook05_diff = {'Mathare': 9.6, 'Kibera': 8.5, 'Mukuru': -0.7}
for name, (lon, lat) in hotspots.items():
    cell = nearest_cell(lon, lat)
    diff = cell['riverside_pct'] - cell['rest_pct']
    print(f"{name:>10} {cell['riverside_pct']:>11.1f}% {cell['rest_pct']:>14.1f}% {diff:>+7.1f}pp   (nb05: {notebook05_diff[name]:+.1f}pp)")

  Location  Riverside %  Rest-of-cell %     Diff   (notebook 05: 1km-radius diff)
   Mathare       100.0%           77.9%   +22.1pp   (nb05: +9.6pp)
    Kibera       100.0%           99.3%    +0.7pp   (nb05: +8.5pp)
    Mukuru        94.1%           96.8%    -2.7pp   (nb05: -0.7pp)


## Visualize

In [6]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(builtup, builtup_vis, '2024 built-up')
Map.addLayer(riverside_mask.selfMask(), {'palette': ['cyan']}, 'Riparian buffer (30m)')

top_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]), {'diff_pp': row['diff_pp']})
    for row in rows[:15]
])
Map.addLayer(top_points, {'color': 'yellow'}, 'Top hotspot cells')
Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Summary

**Top hotspot cells** (578 of 3016 grid cells had enough pixels on both sides to compare;
top 15 by riverside-vs-rest diff):

| lon | lat | riverside % | rest % | diff |
|---|---|---|---|---|
| 36.7960 | -1.2815 | 86.7% | 30.2% | +56.4pp |
| 36.9173 | -1.2906 | 78.0% | 23.2% | +54.8pp |
| 36.7960 | -1.2499 | 96.1% | 43.8% | +52.3pp |
| 36.8005 | -1.2137 | 76.5% | 26.2% | +50.3pp |
| 36.9308 | -1.1686 | 82.1% | 40.5% | +41.5pp |
| 36.8140 | -1.2770 | 93.3% | 53.9% | +39.4pp |
| 37.0610 | -1.2772 | 79.5% | 40.7% | +38.8pp |
| 37.0745 | -1.2908 | 53.0% | 14.9% | +38.1pp |
| 36.8904 | -1.2002 | 78.6% | 42.1% | +36.5pp |
| 36.8140 | -1.2725 | 100.0% | 65.4% | +34.6pp |
| 36.8904 | -1.2364 | 57.7% | 23.4% | +34.3pp |
| 36.9757 | -1.2590 | 74.4% | 40.4% | +34.0pp |
| 36.9308 | -1.2409 | 99.7% | 69.0% | +30.7pp |
| 36.9757 | -1.2229 | 78.9% | 49.4% | +29.5pp |
| 36.9308 | -1.2183 | 100.0% | 70.6% | +29.4pp |

These are stronger differentials (+30 to +56pp) than either literature-known hotspot showed in
notebook 05 — genuine candidates for riverside encroachment surfaced without any prior knowledge
of where to look. Reported here as coordinates for further inspection, not asserted as named
settlements — no ground-truthing or reverse-geocoding was done.

**Cross-check against notebook 05's known hotspots:**

| Location | Riverside % | Rest-of-cell % | Diff (this notebook) | Diff (notebook 05, 1km radius) |
|---|---|---|---|---|
| Mathare | 100.0% | 77.9% | **+22.1pp** | +9.6pp |
| Kibera | 100.0% | 99.3% | **+0.7pp** | +8.5pp |
| Mukuru | 94.1% | 96.8% | -2.7pp | -0.7pp |

**What this reveals:** Mathare shows an even sharper contrast at the fine 500m/within-cell grain
than at 1km radius — its riverside crowding is a genuine sharp local edge. Kibera shows nearly no
local contrast here, despite notebook 05's clear +8.5pp signal at 1km radius — because Kibera's
immediate 500m neighborhood is already saturated built-up on both sides of the 30m line. There's
no local edge for a within-cell method to find; Kibera's riverside character only shows up
against the *wider* city fabric, not its own immediate surroundings.

**Conclusion: the two methods are complementary, not redundant, and neither alone is sufficient.**
A within-cell scan (this notebook) finds sharp built/unbuilt boundaries along rivers with no
prior knowledge required, but misses settlements that are uniformly dense on both sides of the
buffer line. A wider-radius comparison (notebook 05) catches those, but only where you already
know to look. **Decision: a real monitoring tool should run both, not choose one.**

**Caveats:**
- Grid resolution (500m) and the minimum-pixel filter (20px/side, ~2000m² minimum) are analytical
  choices with real, demonstrated consequences (the Kibera discrepancy above) — not free
  parameters to tune away without acknowledging what they trade off.
- No deduplication/clustering of adjacent hot cells into single "sites" — some of the top 15 may
  be neighboring cells along the same real encroachment stretch, reported here as separate rows.
- Same caveats as notebook 05: single-year 2024 snapshot (not change detection — can't say
  encroachment is *worsening*, only where it currently is), HydroSHEDS river completeness for the
  smallest tributaries, 30m buffer is an analytical choice not a legal riparian-reserve
  determination.